##### Imports and setup

In [1]:
# setup the notebook environment
!pixi install
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import logging
logging.getLogger("pypsa.network.descriptors").setLevel(logging.ERROR)

def ensure_project_root_on_path(marker_dir: str = "modules") -> str:
    """Find the nearest ancestor folder containing `marker_dir`, add it to
    sys.path (front) and return its path. Raises RuntimeError if not found.
    """
    cwd = Path.cwd().resolve()
    # If cwd contains marker_dir, use cwd
    if (cwd / marker_dir).exists():
        p = str(cwd)
        if p not in sys.path:
            sys.path.insert(0, p)
        return p
    # Walk parents
    for parent in cwd.parents:
        if (parent / marker_dir).exists():
            p = str(parent)
            if p not in sys.path:
                sys.path.insert(0, p)
            return p
    raise RuntimeError(f"Could not find '{marker_dir}' directory in {cwd} or any parent")


project_root = ensure_project_root_on_path()
print("Added project root to sys.path:", project_root)

# imports and loading data
from modules.analysis_toolkit.helpers.config.constants import CAPTURE_RATE_IC
from modules.analysis_toolkit.helpers.index_finder import GeoOptions
from modules.analysis_toolkit.analyzer import ResultsComputer, GeoOptions, format_result
from modules.analysis_toolkit.helpers.plotting import TimeSeriesPlot, HistogramPlot, BarChartPlot, WaterfallPlot
import pandas as pd

study_years = [2030, 2040]
rc = {year: ResultsComputer(year=year, apply_snapshot_filter=False) for year in study_years}
weights = rc[2030].ns.get_iem_dispatch().snapshot_weightings["generators"]

✔ The default environment has been installed.
Added project root to sys.path: /Users/rca/PycharmProjects/NGV-FBMC


INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Status Quo (SQ) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Integrated Energy Market (IEM) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Flow-based market coupling (FBMC) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'SQ (2030) - redispatch' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, sto

This notebook computes the key indicators reported in [Model run validation checks](https://docs.google.com/document/d/165EH2i8Fgtec260gfUZL6zXfYESi6Uwv6g7zTVdtgiU/edit?pli=1&tab=t.0) to check the consistency of the model runs and the results. The indicators are computed for both the dispatch and redispatch results, and can be compared to the values reported in FES25 or other reference documents.

# Dispatch

## Total GB demand
As reference, FES25 uses the following total GB demand:
* 339 TWh in 2030
* 564 TWh in 2040

In [2]:
pd.concat(
    [
        rc[year].withdrawal.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')") \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method withdrawal
Calling method withdrawal


year,2030,2040
scenario,,
sq,350.6,614.1
iem,350.8,614.3
iem_fb,350.7,615.2
diff: iem-sq,0.1,0.2
diff: iemfb-iem,-0.0,0.9


## Total GB supply

In [3]:
pd.concat(
    [
        rc[year].supply.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')") \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method supply
Calling method supply


year,2030,2040
scenario,,
sq,404.1,694.7
iem,404.1,694.9
iem_fb,403.1,691.6
diff: iem-sq,0.0,0.2
diff: iemfb-iem,-1.0,-3.3


## Breakdown of GB generation per technology

In [4]:
pd.concat(
    [
        rc[year].supply.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')")  \
            .groupby("carrier").sum() \
            .div(1e6) \
            .round(3)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method supply
Calling method supply


year                        2030                                 \
scenario                      sq      iem   iem_fb diff: iem-sq   
carrier                                                           
Battery Storage            6.164    6.271    6.234        0.107   
Onshore Wind              69.641   69.635   69.650       -0.005   
gas-ccgt                  29.704   29.604   28.639       -0.100   
gas-ocgt                   0.000    0.000    0.000        0.000   
geothermal                 0.063    0.063    0.063        0.000   
h2-ccgt                    0.000    0.000    0.000        0.000   
home battery discharger    6.072    6.074    6.021        0.002   
hydro-phs                  3.220    3.223    3.254        0.003   
hydro-reservoir            4.695    4.695    4.696       -0.000   
load                       0.000    0.000    0.000        0.000   
nuclear                   20.017   20.017   20.017        0.000   
offwind-dc-fl-oh         197.744  197.735  197.726       -0.009   
solar-pv-utility          38.365   38.380   38.383        0.015   
solid biomass             28.442   28.437   28.433       -0.005   
waste                      0.000    0.000    0.000        0.000   

year                                        2040                    \
scenario                diff: iemfb-iem       sq      iem   iem_fb   
carrier                                                              
Battery Storage                  -0.037    2.068    2.074    2.485   
Onshore Wind                      0.014   90.419   90.418   89.289   
gas-ccgt                         -0.965    5.800    5.780    5.719   
gas-ocgt                         -0.000      NaN      NaN      NaN   
geothermal                        0.000    0.019    0.019    0.020   
h2-ccgt                          -0.000    3.724    3.748    4.625   
home battery discharger          -0.054   76.447   76.644   74.396   
hydro-phs                         0.032    3.669    3.641    4.170   
hydro-reservoir                   0.001    4.704    4.704    4.678   
load                             -0.000    0.000    0.000    0.000   
nuclear                          -0.000   41.960   41.960   41.960   
offwind-dc-fl-oh                 -0.009  372.181  372.187  371.497   
solar-pv-utility                  0.004   81.875   81.870   81.013   
solid biomass                    -0.004   11.735   11.731   11.600   
waste                            -0.000    0.141    0.163    0.163   

year                                                  
scenario                diff: iem-sq diff: iemfb-iem  
carrier                                               
Battery Storage                0.006           0.411  
Onshore Wind                  -0.001          -1.130  
gas-ccgt                      -0.020          -0.061  
gas-ocgt                         NaN             NaN  
geothermal                     0.000           0.002  
h2-ccgt                        0.023           0.877  
home battery discharger        0.198          -2.248  
hydro-phs                     -0.028           0.529  
hydro-reservoir               -0.000          -0.026  
load                           0.000           0.000  
nuclear                       -0.000           0.000  
offwind-dc-fl-oh               0.006          -0.690  
solar-pv-utility              -0.005          -0.857  
solid biomass                 -0.004          -0.131  
waste                          0.021          -0.000

### Interconnector capacity

In [5]:
ic_capacity = pd.concat(
    [
        rc[year].ns.get_iem_dispatch().links.query(
            "carrier == 'DC' and (bus0.str.contains('GB ') or bus1.str.contains('GB ')) and ~name.str.contains('relation')"
        ) \
        .p_nom
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
ic_capacity.loc["Total", :] = ic_capacity.sum(axis=0)

In [6]:
ic_capacity

year,2030,2040
name,,
Aminth,1400.0,1400.0
BritNed,1200.0,1200.0
Continental Link,0.0,1800.0
Cronos,0.0,1400.0
East-West,585.0,585.0
ElecLink,1000.0,1000.0
FAB Link,0.0,1250.0
Greenlink (Greenwire),524.0,524.0
Gridlink,0.0,1250.0


## GB net position

In [7]:
pd.concat(
    [
        rc[year].energy_balance.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .drop("Line", level="component") \
            .drop(["DC", "DC_OH"], level="carrier") \
            .query("bus.str.contains('GB ')") \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method energy_balance
Calling method energy_balance


year,2030,2040
scenario,,
sq,53.5,80.6
iem,53.4,80.6
iem_fb,52.4,76.4
diff: iem-sq,-0.1,-0.0
diff: iemfb-iem,-1.0,-4.2


## GB prices

In [8]:
pd.concat(
    [
        rc[year].prices.compare_dispatch(bus_carrier=["AC", "AC_OH"]) \
            .query("name.str.contains('GB ')") \
            .describe() \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method prices
Calling method prices


year      2030                                            2040               \
scenario    sq   iem iem_fb diff: iem-sq diff: iemfb-iem    sq   iem iem_fb   
count     24.0  24.0   24.0         24.0            24.0  24.0  24.0   24.0   
mean      45.3  45.5   47.5          0.2             2.0  17.3  17.6   18.0   
std        2.5   2.7    3.4          1.3             2.0   1.2   1.7    1.3   
min       34.4  34.5   34.7         -2.4            -0.2  13.4  13.6   13.9   
25%       45.3  45.4   46.0         -0.2             0.1  17.2  17.3   18.0   
50%       45.6  45.8   47.7          0.2             1.8  17.5  17.6   18.3   
75%       46.0  46.3   49.3          0.5             2.9  17.7  17.8   18.4   
max       48.7  50.5   51.3          4.1             6.4  20.4  23.9   20.9   

year                                   
scenario diff: iem-sq diff: iemfb-iem  
count            24.0            24.0  
mean              0.3             0.4  
std               0.8             1.7  
min              -0.2            -6.0  
25%               0.1             0.4  
50%               0.1             0.6  
75%               0.2             0.9  
max               3.5             4.1

## Congestion revenue

In [9]:
pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum(axis=0) \
            .div(1e6) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/133461534.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/133461534.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,1.3,-63.3
diff: iemfb-iem,-9.0,-2.2
iem,3803.2,3105.7
iem_fb,3794.2,3103.5
sq,3802.0,3169.0


### Breakdown of merchant congestion revenue per interconnector

In [10]:
pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .div(1e6) \
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/429633954.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/429633954.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year                          2030                                       \
scenario              diff: iem-sq diff: iemfb-iem    iem iem_fb     sq   
name                                                                      
Aminth                        -1.7            -0.1  436.5  436.4  438.1   
BritNed                        0.1            -0.8  492.0  491.2  491.9   
Continental Link               0.0             0.0    0.0    0.0    0.0   
Cronos                         0.0             0.0    0.0    0.0    0.0   
East-West                      0.4             0.1   51.9   51.9   51.4   
ElecLink                      -0.6            -2.4  342.0  339.6  342.6   
FAB Link                       0.0             0.0    0.0    0.0    0.0   
Greenlink (Greenwire)          0.4             0.1   46.4   46.6   46.0   
Gridlink                       0.0             0.0    0.0    0.0    0.0   
IFA                           -1.7            -2.6  683.9  681.4  685.6   
IFA2                          -0.7            -1.6  342.0  340.4  342.6   
Kulizumboo                     0.0             0.0    0.0    0.0    0.0   
LirIC                          0.0             0.0    0.0    0.0    0.0   
MARES                          0.6             0.1   66.5   66.5   65.9   
Moyle                          0.8             0.0   88.6   88.6   87.8   
NS Link (NSL)                  5.9            -0.8  404.0  403.3  398.2   
Nautilus                       0.0             0.0    0.0    0.0    0.0   
Nemo                          -0.6            -1.1  413.0  411.9  413.6   
NeuConnect                     0.0             0.0    0.0    0.0    0.0   
NorthConnect                   0.0             0.0    0.0    0.0    0.0   
SENECA                         0.0             0.0    0.0    0.0    0.0   
Viking Link                   -1.7            -0.1  436.5  436.4  438.2   

year                          2040                                       
scenario              diff: iem-sq diff: iemfb-iem    iem iem_fb     sq  
name                                                                     
Aminth                        -7.9            -0.3  103.6  103.3  111.5  
BritNed                       -2.2             1.1  187.9  189.0  190.1  
Continental Link              -8.1             4.4  145.3  149.7  153.4  
Cronos                        -3.0             0.4  224.2  224.6  227.2  
East-West                      0.7            -0.2   47.2   46.9   46.5  
ElecLink                      -1.9            -2.2  125.0  122.8  127.0  
FAB Link                      -2.5            -4.3  156.3  152.0  158.8  
Greenlink (Greenwire)          0.6            -0.2   42.3   42.1   41.6  
Gridlink                      -2.5             0.4  156.3  156.7  158.8  
IFA                           -3.9            -1.1  250.1  249.0  254.0  
IFA2                          -2.4            -3.2  125.0  121.8  127.4  
Kulizumboo                    -1.6            -3.5   87.5   84.0   89.1  
LirIC                          0.8            -0.3   56.4   56.2   55.6  
MARES                          0.9            -0.3   60.5   60.2   59.6  
Moyle                          1.2             0.1   76.8   76.9   75.6  
NS Link (NSL)                 -6.5             3.6  119.5  123.1  126.0  
Nautilus                      -3.1             0.8  240.2  241.0  243.2  
Nemo                          -2.0            -3.5  160.1  156.7  162.1  
NeuConnect                    -3.8             4.1  328.5  332.5  332.3  
NorthConnect                  -6.3             1.1  113.0  114.1  119.3  
SENECA                        -2.3             1.3  196.4  197.7  198.6  
Viking Link                   -7.9            -0.3  103.6  103.3  111.5

## SEW change

In [11]:
pd.concat(
    [
        rc[year].welfare_system.compare_dispatch()
            .round(1) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
) # already in M€

Calling method welfare_system
Calling method welfare_system


year                   2030                                                   \
scenario                 sq       iem    iem_fb diff: iem-sq diff: iemfb-iem   
producer surplus   193490.7  193770.1  193790.8        279.5            20.6   
consumer cost     -330626.9 -330764.3 -330798.4       -137.4           -34.1   
congestion income   21860.8   21886.3   21871.8         25.5           -14.5   
storage surplus     38898.6   38735.1   38754.3       -163.5            19.2   
total_welfare      -76376.8  -76372.7  -76381.4          4.2            -8.8   

year                   2040                                                   
scenario                 sq       iem    iem_fb diff: iem-sq diff: iemfb-iem  
producer surplus    96243.4   97251.6   98494.4       1008.3          1242.8  
consumer cost     -200353.2 -201545.7 -202660.3      -1192.5         -1114.7  
congestion income   27815.6   27766.8   27584.8        -48.8          -182.0  
storage surplus     22256.1   22492.8   22502.7        236.8             9.8  
total_welfare      -54038.1  -54034.4  -54078.5          3.7           -44.0

# Redispatch

## Constraint costs

In [12]:
pd.concat(
    [
        rc[year].constraint_costs.compare_redispatch() \
            .mul(weights, level="snapshot", axis=1)
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method constraint_costs
Calling method constraint_costs


year,2030,2040
scenario,,
diff: iem-sq,4.0,-12.2
diff: iemfb-iem,-4.8,-318.4
iem,3984.0,13563.2
iem_fb,3979.2,13244.9
sq,3980.0,13575.4


## Total redispatched energy volume
redispatch + countertrading + load shedding

In [13]:
pd.concat(
    [
        rc[year].constraint_management_volume.compare_redispatch() \
            .mul(weights, level="snapshot", axis=1) \
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method constraint_management_volume
Calling method constraint_management_volume


year,2030,2040
scenario,,
diff: iem-sq,-0.3,0.2
diff: iemfb-iem,-1.6,-2.2
iem,27.6,84.2
iem_fb,26.0,82.1
sq,27.9,84.0


Breakdown of constraint management volumes per technology/interconnector, in percentage:

In [14]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
year=2030
rc[year].constraint_management_volume.compare_redispatch().groupby("scenario", axis=1).sum().groupby("ramp carrier").sum().div(1e6).round(2) \
    .drop(["battery", "PHS"]).sum()

Calling method constraint_management_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/3460189396.py:4: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



scenario
diff: iem-sq       -0.32
diff: iemfb-iem    -1.61
iem                25.29
iem_fb             23.65
sq                 25.62
dtype: float64

In [15]:
df = rc[2030].constraint_management_volume.compare_redispatch().groupby("scenario", axis=1).sum().groupby("ramp carrier").sum()
(df / df.sum() * 100).iloc[:, 2:].round(2)  # In percentage

Calling method constraint_management_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/3912863907.py:1: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



scenario,iem,iem_fb,sq
ramp carrier,,,
,0.00,0.00,0.00
0 offwind-dc,3.23,3.12,3.20
0 onwind,3.03,3.22,2.99
0 solar,0.33,0.32,0.33
CCGT,37.55,38.02,37.91
Interconnector ramp down,4.39,4.36,4.34
Interconnector ramp up,29.30,27.82,29.03
OCGT,8.89,8.82,8.84
PHS,7.78,8.24,7.67


### Redispatch

#### Redispatch: ramp up

In [16]:
pd.concat(
    [
        rc[year].redispatched_volume.compare_redispatch()
            .query("carrier.str.contains('ramp up')") \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per GB bus (bus), per technology-direction (ramp carrier), or total (sum):
# .groupby("bus").sum()
# .groupby("ramp carrier").sum()
# .sum()


Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/1384244134.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/1384244134.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,-0.1,-0.0
diff: iemfb-iem,0.0,0.1
iem,5.7,1.0
iem_fb,5.8,1.0
sq,5.9,1.0


#### Redispatch: ramp down

In [17]:
pd.concat(
    [
        rc[year].redispatched_volume.compare_redispatch()
            .query("carrier.str.contains('ramp down')") \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per GB bus (bus), per technology-direction (ramp carrier), or total (sum):
# .groupby("bus").sum()
# .groupby("ramp carrier").sum()
# .sum()

Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/4250467477.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method redispatched_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/4250467477.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,-0.2,0.0
diff: iemfb-iem,-0.7,-0.6
iem,12.6,38.0
iem_fb,11.9,37.4
sq,12.8,38.0


### Countertrading

#### Countertrading: ramp up

In [18]:
pd.concat(
    [
        rc[year].countertraded_volume.compare_redispatch()
            .query("carrier.str.contains('ramp up')") \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per neighboring country bus (bus), interconnector name (name), or total (sum):
# .groupby("bus").sum()
# .groupby("name").sum()
# .sum()

Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/999498334.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/999498334.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,-0.0,0.1
diff: iemfb-iem,-0.9,-1.6
iem,8.1,39.2
iem_fb,7.2,37.6
sq,8.1,39.1


#### Countertrading: ramp down

In [19]:
pd.concat(
    [
        rc[year].countertraded_volume.compare_redispatch()
            .query("carrier.str.contains('ramp down')") \
            .mul(weights, level="snapshot", axis=1)
            .groupby("scenario", axis=1).sum() \
            .sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)
# it is possible to show disaggregated per component or group at one of the following levels:
# per neighboring country bus (bus), interconnector name (name), or total (sum):
# .groupby("bus").sum()
# .groupby("name").sum()
# .sum()

Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/983782068.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method countertraded_volume


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/983782068.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,0.0,0.1
diff: iemfb-iem,-0.1,-0.0
iem,1.2,6.0
iem_fb,1.1,6.0
sq,1.2,5.9


### Load shedding (ramp up only)

In [20]:
pd.concat(
    [
        rc[year].load_shedding_volume.compare_redispatch() \
            .mul(weights, level="snapshot", axis=1) \
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method load_shedding_volume
Calling method load_shedding_volume


year,2030,2040
scenario,,
diff: iem-sq,0.0,-0.0
diff: iemfb-iem,0.0,0.0
iem,0.0,0.0
iem_fb,0.0,0.0
sq,0.0,0.0


## Redispatch cost per MWh, ignoring load shedding

In [21]:
pd.concat(
    [
        rc[year].average_ramping_costs_per_mw.compare_redispatch()
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method average_ramping_costs_per_mw
Calling method average_ramping_costs_per_mw


year                 2030                                              2040  \
scenario               sq    iem iem_fb diff: iem-sq diff: iemfb-iem     sq   
avg_cost_ramp_up    140.9  140.8  146.7         -0.1             5.9   62.3   
avg_cost_ramp_down  144.1  147.8  159.8          3.7            12.0  252.0   

year                                                           
scenario              iem iem_fb diff: iem-sq diff: iemfb-iem  
avg_cost_ramp_up     62.2   65.4         -0.1             3.2  
avg_cost_ramp_down  251.1  246.7         -1.0            -4.3

# PLAYGROUND

In [22]:
# redispatch opex
rc[2040].opex.compare_redispatch().query("carrier.str.contains('ramp')").sum().div(1e6).round(1)

Calling method opex


scenario
sq                 5961.2
iem                5949.3
iem_fb             5861.4
diff: iem-sq        -11.9
diff: iemfb-iem     -88.0
dtype: float64

In [23]:
# net boundary loading (-1, 1, both directions) statistics via describe()
format_result(
    result=rc[2030].boundary_loading.compare_redispatch(which="actual").xs("DIRECT", level="direction").sub(
        rc[2030].boundary_loading.compare_redispatch(which="actual").xs("OPPOSITE", level="direction")
    ),
    sum_by=["scenario", "snapshot", "boundary"],
    factor=1
).unstack(["boundary"]).describe().stack("boundary").sort_index(level="boundary").swaplevel(-1, 0).round(3)

Calling method boundary_loading
Calling method boundary_loading
Grouping by ['scenario', 'snapshot', 'boundary'] aggregates across [].


/Users/rca/PycharmProjects/NGV-FBMC/modules/analysis_toolkit/analyzer.py:27: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.

/Users/rca/PycharmProjects/NGV-FBMC/modules/analysis_toolkit/analyzer.py:28: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_13325/3249623454.py:8: FutureWarning:

The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.



scenario        diff: iem-sq  diff: iemfb-iem       iem    iem_fb        sq
boundary                                                                   
B0       25%          -0.000           -0.000     0.103     0.104     0.103
         50%          -0.000            0.000     0.264     0.264     0.264
         75%           0.000            0.001     0.457     0.458     0.458
         count      8760.000         8760.000  8760.000  8760.000  8760.000
         max           0.024            0.028     0.719     0.719     0.719
         mean         -0.000            0.000     0.286     0.286     0.286
         min          -0.019           -0.073    -0.032    -0.032    -0.033
         std           0.002            0.002     0.200     0.200     0.200
B13      25%          -0.001           -0.002     0.118     0.117     0.118
         50%           0.000           -0.000     0.188     0.187     0.188
         75%           0.001            0.001     0.246     0.246     0.246
         count      8760.000         8760.000  8760.000  8760.000  8760.000
         max           0.118            0.143     0.457     0.482     0.448
         mean          0.000           -0.000     0.164     0.164     0.164
         min          -0.106           -0.167    -0.339    -0.329    -0.338
         std           0.007            0.008     0.120     0.120     0.120
B14      25%          -0.001           -0.001    -0.516    -0.516    -0.516
         50%          -0.000            0.000    -0.467    -0.467    -0.468
         75%           0.001            0.001    -0.408    -0.407    -0.407
         count      8760.000         8760.000  8760.000  8760.000  8760.000
         max           0.104            0.078    -0.265    -0.263    -0.268
         mean         -0.000            0.000    -0.465    -0.465    -0.465
         min          -0.136           -0.086    -0.748    -0.747    -0.748
         std           0.008            0.006     0.083     0.083     0.083
B1a      25%          -0.001           -0.001     0.154     0.153     0.153
         50%          -0.000            0.000     0.276     0.277     0.276
         75%           0.001            0.001     0.393     0.393     0.393
         count      8760.000         8760.000  8760.000  8760.000  8760.000
         max           0.051            0.086     0.679     0.681     0.679
         mean         -0.000            0.000     0.270     0.270     0.270
         min          -0.067           -0.110    -0.143    -0.145    -0.143
         std           0.004            0.006     0.148     0.148     0.148
B2       25%          -0.001           -0.001     0.353     0.353     0.353
         50%          -0.000            0.000     0.659     0.658     0.659
         75%           0.001            0.001     0.859     0.858     0.859
         count      8760.000         8760.000  8760.000  8760.000  8760.000
         max           0.065            0.098     1.000     1.000     1.000
         mean         -0.000            0.000     0.600     0.601     0.601
         min          -0.061           -0.072    -0.143    -0.148    -0.143
         std           0.004            0.006     0.292     0.292     0.292
B3b      25%          -0.001           -0.000     0.215     0.216     0.211
         50%           0.000           -0.000     0.793     0.792     0.792
         75%           0.002            0.003     1.000     1.000     1.000
         count      8760.000         8760.000  8760.000  8760.000  8760.000
         max           0.823            1.968     1.000     1.000     1.000
         mean          0.001            0.001     0.544     0.545     0.543
         min          -0.537           -1.996    -1.000    -1.000    -1.000
         std           0.036            0.093     0.567     0.566     0.568
B4       25%          -0.002           -0.001     0.318     0.317     0.317
         50%          -0.000            0.000     0.562     0.563     0.563
         75%           0.001            0.002   

In [24]:
year = 2030
shedding_volumes = rc[year].load_shedding_volume.compare_redispatch()
shedding_snapshots = shedding_volumes.loc[:, shedding_volumes.sum(axis=0)>0].columns.get_level_values("snapshot")

Calling method load_shedding_volume


In [25]:
shedding_snapshots

DatetimeIndex(['2009-12-19 19:00:00', '2009-01-09 22:00:00',
               '2009-01-09 23:00:00', '2009-01-10 00:00:00',
               '2009-01-10 01:00:00', '2009-01-10 02:00:00',
               '2009-01-10 03:00:00', '2009-01-10 04:00:00',
               '2009-01-10 05:00:00', '2009-01-10 06:00:00',
               '2009-01-10 07:00:00', '2009-01-10 08:00:00',
               '2009-12-19 14:00:00', '2009-12-19 15:00:00',
               '2009-12-19 16:00:00', '2009-12-19 17:00:00',
               '2009-12-19 18:00:00', '2009-12-19 19:00:00',
               '2009-12-19 19:00:00', '2009-01-09 22:00:00',
               '2009-01-09 23:00:00', '2009-01-10 00:00:00',
               '2009-01-10 01:00:00', '2009-01-10 02:00:00',
               '2009-01-10 03:00:00', '2009-01-10 04:00:00',
               '2009-01-10 05:00:00', '2009-01-10 06:00:00',
               '2009-01-10 07:00:00', '2009-01-10 08:00:00',
               '2009-12-19 14:00:00', '2009-12-19 15:00:00',
               '2009-12-

In [26]:
available_redispatch_capacity = rc[2030].available_redispatch_capacity_mw.sq_redispatch().filter(regex='^(?!.*EU)', axis=1)

Calling method available_redispatch_capacity_mw


In [27]:
available_redispatch_capacity.loc[shedding_snapshots].filter(like="ramp up").sort_values(by=shedding_snapshots[0], axis=1, ascending=False)

ValueError: The index label '2009-12-19 19:00:00' is not unique.

In [ ]:
used_redispatch_capacity = rc[2030].used_redispatch_capacity_pu.sq_redispatch().filter(regex='^(?!.*EU)', axis=1)

In [ ]:
used_redispatch_capacity.loc[shedding_snapshots[0]].filter(like="ramp up").sort_values().head(20).round(2)#(by=shedding_snapshots[0], axis=1, ascending=True)

In [ ]:
net_position_gb_buses = rc[2030].net_position.compare_dispatch().groupby("bus").sum().xs(shedding_snapshots[0], level="snapshot", axis=1)

In [ ]:
net_position_gb_buses.filter(like="GB ", axis=0).round()